# Amplitude Amplification 

Amplitude amplification is a generalization of Grover’s search. It allows us to increase the probability of a “good” (marked) result if there is already a quantum algorithm $A$ available that finds a solution with some probability $p$.

If the algorithm $A$ sometimes gives the correct answer, then with the help of special reflection operators we can “rotate” the quantum state so that the probability of obtaining the correct answer becomes larger.


## Basic idea

We have a unitary operator $A$ that prepares the state $|\psi\rangle = A|0\rangle$ from the initial state $|0\rangle.$

In this state, part of the amplitudes lie on the "marked" states and part on the "unmarked" states.

- $|\psi_g\rangle$ - normalized subspace of the "marked" states,
- $|\psi_b\rangle$ - normalized subspace of the "unmarked" states.

Then

$$
|\psi\rangle = \sqrt{p}\,|\psi_g\rangle + \sqrt{1-p}\,|\psi_b\rangle,
$$

where $p$ is the probability that a measurement gives some marked state.

We introduce the angle $\theta$ such that

$$
\sin^2\theta = p,
\qquad 0<\theta\le \frac{\pi}{2}.
$$

Then we can write:

$$
|\psi\rangle = \sin\theta\,|\psi_g\rangle + \cos\theta\,|\psi_b\rangle.
$$

This is a 2-dimensional geometry in the plane spanned by $|\psi_g\rangle$ and $|\psi_b\rangle$.

## Reflections

Amplitude amplification is based on two reflection operators.

### Oracle reflection

We define the phase oracle:

$$
S_f |x\rangle = (-1)^{f(x)} |x\rangle.
$$

It is responsible for making sure that
- the "marked" states get phase $-1$,
- the remaining states stay unchanged.

### Reflection about the initial state

We define the reflection about $|0\rangle$:

$$
S_0 = I - 2|0\rangle\langle 0|.
$$

The reflection about $|\psi\rangle = A|0\rangle$ can be implemented as:

$$
S_\psi = A S_0 A^\dagger.
$$

We define the operator:

$$
Q = -A S_0 A^\dagger S_f.
$$

This is a composition of two reflections.  
Geometrically, the composition of two reflections corresponds to a rotation.

## Rotation interpretation

The initial state:

$$
|\psi\rangle
=
\sin(\theta)\,|\psi_{\text{good}}\rangle
+
\cos(\theta)\,|\psi_{\text{bad}}\rangle.
$$

After $k$ iterations we obtain:

$$
Q^k |\psi\rangle
=
\sin\big((2k+1)\theta\big)\,|\psi_{\text{good}}\rangle
+
\cos\big((2k+1)\theta\big)\,|\psi_{\text{bad}}\rangle.
$$

Thus, the probability that a marked state will be measured after $k$ iterations is:

$$
p_k = \sin^2\big((2k+1)\theta\big).
$$

The maximum probability is reached when:

$$
(2k+1)\theta \approx \frac{\pi}{2}.
$$

From this we obtain the approximate optimal number of iterations:

$$
k \approx \left\lfloor \frac{\pi}{4\theta} - \frac{1}{2} \right\rfloor.
$$

If $k$ is too large, then $(2k+1)\theta$ exceeds $\pi/2$ and the probability starts to decrease.  

## Relation to Grover’s algorithm

Grover’s search is a special case of amplitude amplification, where:

- $A$ prepares an equal superposition,
- $S_f$ marks the searched element,
- $Q$ is repeated the optimal number of times.

In [1]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit import transpile

# The marked state is <101|

qc = QuantumCircuit(3, 3)

# Oracle implementation (general phase flip for a specific bitstring)
def phase_flip(qc, qubits, bitstring):
    q0, q1, q2 = qubits
    bits = [int(b) for b in bitstring]

    # Transform the target state into |111> (apply X where bit = 0)
    if bits[0] == 0: qc.x(q0)
    if bits[1] == 0: qc.x(q1)
    if bits[2] == 0: qc.x(q2)

    # CCZ implemented via H + CCX + H on the last qubit
    qc.h(q2)
    qc.ccx(q0, q1, q2)
    qc.h(q2)

    # Undo the previous X gates
    if bits[0] == 0: qc.x(q0)
    if bits[1] == 0: qc.x(q1)
    if bits[2] == 0: qc.x(q2)

# Prepare an arbitrarily chosen non-uniform superposition
qc.ry(0.9, 0)
qc.ry(1.7, 1)
qc.ry(1.2, 2)

# Number of iterations
num_iterations = 2

for i in range(num_iterations):

    # Sf: oracle 
    phase_flip(qc, [0, 1, 2], "101")

    # A†: inverse of the preparation (reversed order and negative angles)
    qc.ry(-1.2, 2)
    qc.ry(-1.7, 1)
    qc.ry(-0.9, 0)

    # S0: phase inversion only for |000>
    
    qc.x([0, 1, 2])

    # CCZ on |111>
    qc.h(2)
    qc.ccx(0, 1, 2)
    qc.h(2)

    qc.x([0, 1, 2])   # back to the original basis

    # A: forward again
    qc.ry(0.9, 0)
    qc.ry(1.7, 1)
    qc.ry(1.2, 2)

qc.measure([0, 1, 2], [0, 1, 2])

print(qc.draw())

sim = AerSimulator()
tqc = transpile(qc, sim)
result = sim.run(tqc, shots=2000).result()
counts = result.get_counts()

print("Measurement distribution:", counts)

     ┌─────────┐          ┌──────────┐   ┌───┐                   ┌───┐»
q_0: ┤ Ry(0.9) ├───────■──┤ Ry(-0.9) ├───┤ X ├────────────────■──┤ X ├»
     ├─────────┤┌───┐  │  └──┬───┬───┘┌──┴───┴───┐┌───┐       │  ├───┤»
q_1: ┤ Ry(1.7) ├┤ X ├──■─────┤ X ├────┤ Ry(-1.7) ├┤ X ├───────■──┤ X ├»
     ├─────────┤├───┤┌─┴─┐   ├───┤    ├──────────┤├───┤┌───┐┌─┴─┐├───┤»
q_2: ┤ Ry(1.2) ├┤ H ├┤ X ├───┤ H ├────┤ Ry(-1.2) ├┤ X ├┤ H ├┤ X ├┤ H ├»
     └─────────┘└───┘└───┘   └───┘    └──────────┘└───┘└───┘└───┘└───┘»
c: 3/═════════════════════════════════════════════════════════════════»
                                                                      »
«     ┌─────────┐                     ┌──────────┐   ┌───┐                   »
«q_0: ┤ Ry(0.9) ├──────────────────■──┤ Ry(-0.9) ├───┤ X ├────────────────■──»
«     ├─────────┤   ┌───┐          │  └──┬───┬───┘┌──┴───┴───┐┌───┐       │  »
«q_1: ┤ Ry(1.7) ├───┤ X ├──────────■─────┤ X ├────┤ Ry(-1.7) ├┤ X ├───────■──»
«     └──┬───┬──┘┌──┴───┴──┐┌───┐┌─┴